**Notebook 06 — Graphe Social Instagram**

In [1]:
import json
import os
import re
import sys

import pandas as pd

# ── Config centrale ────────────────────────────────────────────────────────────
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '../../..')))
from config import WAREHOUSE, RAW_DATA, CLOSE_FRIENDS, CLOSE_FRIENDS_MULTIPLIER, MIN_MESSAGES


In [2]:
# ─── CHEMINS ─────────────────────────────────────────────────────────────────
# WAREHOUSE et RAW_DATA viennent de config.py
INBOX = os.path.join(RAW_DATA, 'INSTAGRAM', 'your_instagram_activity', 'messages', 'inbox')
os.makedirs(WAREHOUSE, exist_ok=True)

print('Warehouse:', WAREHOUSE)
print('Inbox conversations:', len(os.listdir(INBOX)) if os.path.exists(INBOX) else 'N/A')


BASE: /opt/spark
Inbox conversations: 402


In [5]:
# ─── CLOSE FRIENDS ───────────────────────────────────────────────────────────
# Chargés depuis config.py — modifier config.py à la racine du projet
print(f"Close friends configurés : {len(CLOSE_FRIENDS)}")


Close friends configurés : 28


In [6]:
# ─── PARSING DE L'INBOX ──────────────────────────────────────────────────────
def _parse_conversation(conv_dir: str, folder: str) -> dict | None:
    msg_files = sorted(
        [f for f in os.listdir(conv_dir) if f.startswith("message_") and f.endswith(".json")],
        key=lambda f: int(re.search(r'(\d+)', f).group(1)),
        reverse=True,
    )
    if not msg_files:
        return None

    # Lire le premier fichier pour les métadonnées (participants)
    with open(os.path.join(conv_dir, msg_files[0]), encoding="utf-8") as f:
        data = json.load(f)

    if len(data.get("participants", [])) != 2:
        return None

    # Sommer les messages de TOUS les fichiers (cap à 10 000 msgs/fichier)
    msg_count = 0
    for fname in msg_files:
        with open(os.path.join(conv_dir, fname), encoding="utf-8") as f:
            msg_count += len(json.load(f).get("messages", []))

    if msg_count < MIN_MESSAGES:
        return None

    label = re.split(r'_\d{10,}', folder)[0].lower()
    node_id = folder.lower()
    is_close = (label in CLOSE_FRIENDS) or (node_id in CLOSE_FRIENDS)

    return {"node_id": node_id, "label": label, "message_count": msg_count, "in_close_friends": is_close}


records = []
for folder in os.listdir(INBOX):
    conv_dir = os.path.join(INBOX, folder)
    if not os.path.isdir(conv_dir):
        continue
    result = _parse_conversation(conv_dir, folder)
    if result:
        records.append(result)

df = pd.DataFrame(records)
df["weight"] = df.apply(
    lambda row: row["message_count"] * CLOSE_FRIENDS_MULTIPLIER if row["in_close_friends"] else float(row["message_count"]),
    axis=1,
)
df = df.sort_values("weight", ascending=False).reset_index(drop=True)

print(f"Conversations retenues (>= {MIN_MESSAGES} msgs) : {len(df)}")
print(f"Close friends : {df['in_close_friends'].sum()}")
print()
print(df[["label", "message_count", "in_close_friends", "weight"]].head(30).to_string())

Conversations retenues (>= 5 msgs) : 128
Close friends : 29

               label  message_count  in_close_friends    weight
0                lou          66685              True  133370.0
1               nana          34809              True   69618.0
2              pilou          24502              True   49004.0
3             maelle          22257              True   44514.0
4                jen          16506              True   33012.0
5             djyoyo          12506              True   25012.0
6               evan          11607              True   23214.0
7              laura          10762              True   21524.0
8             loulou           9920              True   19840.0
9              alice           9726              True   19452.0
10              gabi           6502              True   13004.0
11          3li0tttt           5306              True   10612.0
12            mylene           3695              True    7390.0
13             laure           4974        

In [7]:
# ─── SAUVEGARDE ──────────────────────────────────────────────────────────────
out_path = os.path.join(WAREHOUSE, "social_graph.parquet")
df.to_parquet(out_path, index=False)
print(f"Sauvegardé : {out_path} ({len(df)} lignes, {df['message_count'].sum():,} msgs total)")

Sauvegardé : /opt/spark/warehouse/social_graph.parquet (128 lignes, 267,499 msgs total)
